[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonardijulia/AGAI_practice/blob/main/AlphaEarth_GEE/AE_Exercises.ipynb)

# Imports

In [23]:
import ee
import geemap
import numpy as np
import pandas as pd
from google.colab import drive

import ipywidgets as widgets
from IPython.display import display

# Authenticate to GEE and connect to GDrive
To Authenticate and Initialize your GEE project and thus have access to the GEE data catalog you need to first have
  1) A google account
  2) A GEE account with a valid non commercial project

In [2]:
ee.Authenticate()
ee.Initialize(project='lithe-timer-424608-c4')

drive.mount("/content/drive/")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


# 1. Explore the AlphaEarth collection

In [3]:
# Access the Satelite Embedding dataset
collection = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

# Let's see what an ImageCollection object contains and display the metadata of the first element fetched
# NOTE:
# print(obj) reads client-side only (showing a raw, unreadable server-pointer JSON).
# display(obj) automatically fetches server-side data into a rich Jupyter notebook view.
display(collection.limit(1))


More technical details about the data: https://developers.google.com/earth-engine/guides/aef_on_gcs_readme

In [9]:
# Select a region
polygon = ee.Geometry.Polygon([
    [
        [20.7130, 52.3946],  # Northwest corner
        [21.4155, 52.3946],  # Northeast corner
        [21.4155, 52.0775],  # Southeast corner
        [20.7130, 52.0775],  # Southwest corner
        [20.7130, 52.3946]   # Close the polygon ring back at the NW corner
    ]
  ]) # Warsaw, Poland (feel free to change the region and explore your favorite place :))

# Let's see the embedding fields for two consecutive years and compare them
# 2023 embeddings
image1 = collection \
      .filterDate('2023-01-01', '2024-01-01') \
      .filterBounds(polygon) \
      .mosaic() \
      .clip(polygon) # We can also clip the image
# We get multiple Satellite Embedding tiles covering the ROI
# We can use .mosaic() to combine multiple tiles into a single image

# 2024 embeddings
image2 = collection \
      .filterDate('2024-01-01', '2025-01-01') \
      .filterBounds(polygon) \
      .mosaic() \
      .clip(polygon)

In [10]:
display(image1.bandNames())

In [11]:
# The embedding field contains 64 bands, so it's not easy to visualize all teh information
# We can only see a combination of three bands at a time!
# Let's pick three bands to visualize three axes of the embedding space as an "RGB" image.
visParams = {'min'   : -0.3,
             'max'   : 0.3,
             'bands' : ['A08', 'A62', 'A16']}

### Visualize

In [13]:
# One tile ~ 164x164 km
map = geemap.Map(basemap='HYBRID', center=[52.2298, 21.0118], zoom=10)
map.addLayer(image1, visParams, '2023 EMB')
map.addLayer(image2, visParams, '2024 EMB')
map.addLayer(polygon, {'color': 'green', 'alpha': 0.5}, 'Warsaw Polygon')
display(map)

Map(center=[52.2298, 21.0118], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

### Compute the similarity between embedding vectors

The embeddings are **unit-length**, meaning they have a magnitude of 1 and do not require any additional normalization, and are distributed across the unit sphere. With this property, computing the dot product will simplify to the cosine of the angle between the embedding vectors.

**Geometric formula of the dot product**:

$$\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\| \|\mathbf{v}\| \cos\theta$$

$\|\mathbf{u}\| = 1$ and $\|\mathbf{v}\| = 1$.

In [14]:
# .multiply() does element-wise multiplication for each band (A1*B1, A2*B2, etc.).
# .reduce(ee.Reducer.sum()) sums all those band products together to complete the dot product.
dot_prod = image1 \
    .multiply(image2) \
    .reduce(ee.Reducer.sum())

map.addLayer(
    dot_prod,
    {'min': 0, 'max': 1, 'palette': ['white', 'black']},
    'Similarity between years (brighter = less similar)'
)
display(map)

Map(bottom=86613.0, center=[52.23368724057246, 21.0497828989223], controls=(WidgetControl(options=['position',…

# 2. Clustering

We can also try and understand how the model has learnt the spatial and temporal variability of the landscape, by visualizing clusters of similar embeddings.

In [15]:
# Let's sample 1000 embeddings from our region of interest (year 2023)
n_samples = 1000
training = image1.sample(**{
    'region': polygon,
    'scale': 10,
    'numPixels': n_samples,
    'seed': 42
})

In [16]:
# Explore the values of the first training sample
first_sample = training.first()
bands = first_sample.getInfo()['properties']
vector = np.array(list(bands.values()), dtype=float)

# Let's check out if the vector has a unit length
magnitude = np.linalg.norm(vector)
print(f"The embedding vector length fromthe origin to the values of the vector is: {magnitude}")

The embedding vector length fromthe origin to the values of the vector is: 1.0014114706445898


Google employs quantization mapping to compress raw deep learning embeddings before storing them as Cloud Optimized GeoTIFFs, which causes slight precision loss and accumulated rounding errors when fetched via Python. More info: https://developers.google.com/earth-engine/guides/aef_on_gcs_readme

In [17]:
def get_clusters(n_clusters: int):
  clusterer = ee.Clusterer.wekaKMeans(n_clusters).train(training)
  image_clusters = image1.cluster(clusterer)
  return image_clusters

In [18]:
three_clusters = get_clusters(3)
map.addLayer(three_clusters.randomVisualizer().clip(polygon), {}, '3 clusters')
display(map)

Map(bottom=86613.0, center=[52.23368724057246, 21.0497828989223], controls=(WidgetControl(options=['position',…

In [19]:
ten_clusters = get_clusters(10)
map.addLayer(ten_clusters.randomVisualizer().clip(polygon), {}, '10 clusters')
display(map)

Map(bottom=86588.0, center=[52.25470880113083, 21.110217041571687], controls=(WidgetControl(options=['position…

# 3. Supervised Classification

AE embeddings were specifically designed for low-shot learning, meaning that we need a relatively small number of labeled data to achieve high-quality classification results (10s to 100s of samples vs 1000+ for any other decent DL algorithm using raw data).

In [ ]:
#TODO: Manual sampling and training a RF model

# 4. Regression

In [28]:
# Select a new region for AGB estimation
polygon = ee.Geometry.Polygon([[
    [74.322, 14.981],
    [74.322, 14.765],
    [74.648, 14.765],
    [74.648, 14.980]
]])

map = geemap.Map(basemap='SATELLITE', center=[14.765, 74.322], zoom=7)
map.addLayer(polygon, {'color': 'green', 'alpha': 0.5}, 'Polygon')
display(map)

Map(center=[14.765, 74.322], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [30]:
startDate = ee.Date.fromYMD(2022, 1, 1)
endDate = startDate.advance(1, 'year')
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
embeddingsFiltered = embeddings \
  .filter(ee.Filter.date(startDate, endDate)) \
  .filter(ee.Filter.bounds(polygon))

In [32]:
embeddings_projection = ee.Image(embeddingsFiltered.first()).select(0).projection()
embeddings_image = embeddingsFiltered.mosaic() \
  .setDefaultProjection(embeddings_projection)

In [33]:
gedi = ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
def quality_mask(image):
  return image.updateMask(image.select('l4_quality_flag').eq(1)) \
      .updateMask(image.select('degrade_flag').eq(0))

def error_mask(image):
  relative_se = image.select('agbd_se').divide(image.select('agbd'))
  return image.updateMask(relative_se.lte(0.5))

def slope_mask(image):
  glo30 = ee.ImageCollection('COPERNICUS/DEM/GLO30')
  glo30_filtered = glo30 \
    .filter(ee.Filter.bounds(polygon)) \
    .select('DEM')

  dem_proj = glo30_filtered.first().select(0).projection()
  elevation = glo30_filtered.mosaic().rename('dem') \
    .setDefaultProjection(dem_proj)

  slope = ee.Terrain.slope(elevation)
  return image.updateMask(slope.lt(30))

gedi_filtered = gedi \
  .filter(ee.Filter.date(startDate, endDate)) \
  .filter(ee.Filter.bounds(polygon))

gedi_projection = ee.Image(gedi_filtered.first()).select(0).projection()
gedi_processed = gedi_filtered \
  .map(quality_mask) \
  .map(error_mask) \
  .map(slope_mask)

gedi_mosaic = gedi_processed.mosaic() \
  .select('agbd') \
  .setDefaultProjection(gedi_projection)

gediVis = {
  'min': 0,
  'max': 200,
  'palette': ['#edf8fb', '#b2e2e2', '#66c2a4', '#2ca25f', '#006d2c'],
  'bands': ['agbd']
}

map.addLayer(gedi_mosaic, gediVis, 'GEDI AGBD')
display(map)

Map(bottom=120438.0, center=[14.846576365691364, 74.48736694057914], controls=(WidgetControl(options=['positio…

In [34]:
grid_scale = 100
grid_projection = ee.Projection('EPSG:3857').atScale(grid_scale)

stacked = embeddings_image.addBands(gedi_mosaic)
stacked = stacked.resample('bilinear')
stacked_resampled = stacked \
  .reduceResolution({
    'reducer': ee.Reducer.mean(),
    'maxPixels': 1024
  }) \
  .reproject({
    'crs': grid_projection
})

tackedResampled = stacked_resampled \
  .updateMask(stacked_resampled.mask().gt(0))

TypeError: Reducer can only be used as a cast to Reducer. Found <class 'dict'>. If you are trying to pass keyword arguments as a dictionary, use the `**` dictionary unpacking operator (e.g., `myFunction(**kwargs)`).

# 5. Similarity Search

If you want to further explore similarity between locations, check out: https://earthengine-ai.projects.earthengine.app/view/embedding-similarity-search#year=2025